In [1]:
# =========================
# Hypothesis Tests
# Main specification:
# Use the combined modeling dataset + CAR from Y
# =========================

import pandas as pd
import numpy as np

from scipy.stats import pearsonr, spearmanr, ttest_ind, mannwhitneyu
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests


# =========================
# 1. Load data
# =========================
COMBINED_PATH = "model_finance_text_combined.csv"
Y_PATH = "deepseek_Y_quantile_final.csv"

df_combined = pd.read_csv(COMBINED_PATH)
y_full = pd.read_csv(Y_PATH)

print("df_combined shape:", df_combined.shape)
print("y_full shape:", y_full.shape)

print("\ndf_combined columns:")
print(df_combined.columns.tolist())

print("\ny_full columns:")
print(y_full.columns.tolist())


# =========================
# 2. Merge CAR into combined dataset
# =========================
# Keep only columns needed from Y
y_keep = y_full[["ticker", "CAR", "label"]].drop_duplicates().copy()

df = df_combined.merge(
    y_keep,
    on=["ticker", "label"],
    how="left"
)

print("\nMerged hypothesis-test dataset shape:", df.shape)
print("\nMissing CAR count:", df["CAR"].isna().sum())
print("\nLabel counts:")
print(df["label"].value_counts(dropna=False))


# =========================
# 3. Select variables to test
# =========================
# You can edit this list if you want more / fewer variables.

text_vars = [
    "mean_sentiment",
    "std_sentiment",
    "ratio_strong_positive",
    "ratio_strong_negative",
    "ai_topic_loading_sum",
    "ai_keyword_ratio",
    "topic_02_ai_semiconductor_design",
    "topic_09_ai_cloud_infrastructure",
    "topic_11_ai_saas_cybersecurity",
]

finance_vars = [
    "PC1", "PC2", "PC3", "PC4", "PC5", "PC6", "PC7", "PC8"
]

test_vars = text_vars + finance_vars

print("\nVariables to test:")
print(test_vars)


# =========================
# 4. Part A — Continuous tests with CAR
#    Pearson, Spearman, OLS(HC3)
# =========================
continuous_results = []

for var in test_vars:
    tmp = df[[var, "CAR"]].dropna().copy()

    if len(tmp) < 10:
        continue

    # Pearson correlation
    pearson_r, pearson_p = pearsonr(tmp[var], tmp["CAR"])

    # Spearman correlation
    spearman_rho, spearman_p = spearmanr(tmp[var], tmp["CAR"])

    # OLS with robust SE
    X = sm.add_constant(tmp[[var]])
    y = tmp["CAR"]
    ols_model = sm.OLS(y, X).fit(cov_type="HC3")

    coef = ols_model.params[var]
    coef_p = ols_model.pvalues[var]
    t_stat = ols_model.tvalues[var]

    continuous_results.append({
        "variable": var,
        "n": len(tmp),
        "pearson_r": pearson_r,
        "pearson_p": pearson_p,
        "spearman_rho": spearman_rho,
        "spearman_p": spearman_p,
        "ols_coef": coef,
        "ols_t": t_stat,
        "ols_p": coef_p
    })

continuous_df = pd.DataFrame(continuous_results)

# FDR correction for OLS p-values
reject, p_adj, _, _ = multipletests(continuous_df["ols_p"], method="fdr_bh")
continuous_df["ols_p_fdr"] = p_adj
continuous_df["reject_H0_5pct"] = np.where(continuous_df["ols_p"] < 0.05, "Reject", "Fail to reject")
continuous_df["reject_H0_5pct_fdr"] = np.where(continuous_df["ols_p_fdr"] < 0.05, "Reject", "Fail to reject")

continuous_df = continuous_df.sort_values("ols_p")

print("\n=== Continuous tests vs CAR ===")
print(continuous_df.round(4).to_string(index=False))


# =========================
# 5. Part B — Group difference tests
#    Vulnerable vs Resilient
#    Welch t-test + Mann-Whitney U
# =========================
group_results = []

for var in test_vars:
    g_vul = df[df["label"] == "Vulnerable"][var].dropna()
    g_res = df[df["label"] == "Resilient"][var].dropna()

    if len(g_vul) < 5 or len(g_res) < 5:
        continue

    # Welch t-test
    t_stat, t_p = ttest_ind(g_vul, g_res, equal_var=False)

    # Mann-Whitney U
    u_stat, u_p = mannwhitneyu(g_vul, g_res, alternative="two-sided")

    # Mean difference
    mean_vul = g_vul.mean()
    mean_res = g_res.mean()
    diff = mean_vul - mean_res

    group_results.append({
        "variable": var,
        "n_vulnerable": len(g_vul),
        "n_resilient": len(g_res),
        "mean_vulnerable": mean_vul,
        "mean_resilient": mean_res,
        "mean_diff_vul_minus_res": diff,
        "t_stat": t_stat,
        "t_p": t_p,
        "u_stat": u_stat,
        "u_p": u_p
    })

group_df = pd.DataFrame(group_results)

# FDR correction for t-test p-values
reject, p_adj, _, _ = multipletests(group_df["t_p"], method="fdr_bh")
group_df["t_p_fdr"] = p_adj
group_df["reject_H0_5pct"] = np.where(group_df["t_p"] < 0.05, "Reject", "Fail to reject")
group_df["reject_H0_5pct_fdr"] = np.where(group_df["t_p_fdr"] < 0.05, "Reject", "Fail to reject")

group_df = group_df.sort_values("t_p")

print("\n=== Group difference tests: Vulnerable vs Resilient ===")
print(group_df.round(4).to_string(index=False))


# =========================
# 6. Create a cleaner summary table
# =========================
summary_rows = []

# Continuous part
for _, row in continuous_df.iterrows():
    summary_rows.append({
        "test_block": "continuous_vs_CAR",
        "variable": row["variable"],
        "test": "OLS_HC3",
        "statistic": row["ols_t"],
        "p_value": row["ols_p"],
        "p_value_fdr": row["ols_p_fdr"],
        "decision_5pct": row["reject_H0_5pct"],
        "decision_5pct_fdr": row["reject_H0_5pct_fdr"],
        "effect": row["ols_coef"],
        "interpretation": (
            "positive association with CAR" if row["ols_coef"] > 0
            else "negative association with CAR"
        )
    })

# Group difference part
for _, row in group_df.iterrows():
    summary_rows.append({
        "test_block": "group_diff_Vulnerable_vs_Resilient",
        "variable": row["variable"],
        "test": "Welch_t_test",
        "statistic": row["t_stat"],
        "p_value": row["t_p"],
        "p_value_fdr": row["t_p_fdr"],
        "decision_5pct": row["reject_H0_5pct"],
        "decision_5pct_fdr": row["reject_H0_5pct_fdr"],
        "effect": row["mean_diff_vul_minus_res"],
        "interpretation": (
            "higher in Vulnerable" if row["mean_diff_vul_minus_res"] > 0
            else "higher in Resilient"
        )
    })

summary_df = pd.DataFrame(summary_rows)

print("\n=== Hypothesis test summary ===")
print(summary_df.round(4).to_string(index=False))


# =========================
# 7. Save outputs
# =========================
continuous_df.to_csv("hypothesis_tests_continuous_vs_CAR.csv", index=False)
group_df.to_csv("hypothesis_tests_group_diff.csv", index=False)
summary_df.to_csv("hypothesis_tests_summary.csv", index=False)

print("\nSaved files:")
print("  hypothesis_tests_continuous_vs_CAR.csv")
print("  hypothesis_tests_group_diff.csv")
print("  hypothesis_tests_summary.csv")

df_combined shape: (154, 43)
y_full shape: (158, 17)

df_combined columns:
['companyid', 'companyname', 'n_sentences', 'mean_p_positive', 'mean_p_negative', 'mean_p_neutral', 'mean_sentiment', 'std_sentiment', 'pct_negative', 'pct_neutral', 'pct_positive', 'ratio_strong_negative', 'ratio_strong_positive', 'topic_00_energy_utilities', 'topic_01_advertising_marketing', 'topic_02_ai_semiconductor_design', 'topic_03_ai_enterprise_healthcare_services', 'topic_04_financial_reporting_geographic', 'topic_05_financial_reporting_accounting', 'topic_06_government_defense', 'topic_07_project_execution_capital', 'topic_08_china_ai_semiconductor_supply_chain', 'topic_09_ai_cloud_infrastructure', 'topic_10_chinese_gaming_streaming', 'topic_11_ai_saas_cybersecurity', 'ai_topic_loading_sum', 'ai_keyword_count', 'ai_keyword_ratio', 'companyid_bridge', 'ticker', 'company_name', 'gvkey', 'datadate', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'label', 'label_binary']

y_full columns:
['company